In [49]:
import numpy as np
import matplotlib.pyplot as plt

#Voltage differnce
U = 1.0 #vout - vin

#nodes
nodes = np.array([[0.0, 1.0], [-0.5, 0.5], [-0.8, 0.0], [0.6, 0.4],[0.0,-0.2],[1.0,-0.1]])

#connectivity , nodes[i-1]

Elements = np.array([[nodes[0],nodes[1],nodes[3]], [nodes[3],nodes[1],nodes[4]], [nodes[2],nodes[4],nodes[1]], [nodes[4],nodes[5],nodes[3]]])



#number of nodes , elements
n_nodes = len(nodes)    
n_elements = len(Elements)

print("Number of nodes: ", n_nodes)
print("Number of elements: ", n_elements)

print("\n")

print("Elements:")
for i in range(n_elements):
    print(f"Element {i+1}: {Elements[i]}")
print("\n")
"""
#global stiffness matrix
A = np.zeros((n_nodes, n_nodes))
b = np.zeros(n_nodes)
"""



#Single Matrix element from lab 08
def element_matrix(element):

    x = element[:, 0]
    y = element[:, 1]

    S1 = np.array([x[2]-x[1], y[2]-y[1]])
    S2 = np.array([x[0]-x[2], y[0]-y[2]])
    S3 = np.array([x[1]-x[0], y[1]-y[0]])


    A_tot = 0.5 * ((x[1]-x[0])*(y[2]-y[0]) - (x[2]-x[0])*(y[1]-y[0]))


    if A_tot < 0:
        raise ValueError("Nodes are not in counterclockwise order")

    if A_tot == 0:
        raise ValueError("Triangle has zero area")


    #Gradients of the basis functions (𝜑𝑖)

    grad_phi = np.array([[-S1[1], -S2[1], -S3[1]], [ S1[0],  S2[0],  S3[0]]]) / (2 * A_tot)


    Aij = np.zeros((3, 3))
    Aij = A_tot * np.dot(grad_phi.T, grad_phi)

    return Aij,A_tot



def Global_matrix(Elements, n_nodes):
    A = np.zeros((n_nodes, n_nodes))
    b = np.zeros(n_nodes)

    for element in Elements:
        Aij, A_tot = element_matrix(element)
        node_indices = [np.where((nodes == node).all(axis=1))[0][0] for node in element]

        for i in range(3):
            for j in range(3):
                A[node_indices[i], node_indices[j]] += Aij[i, j]

    return A, b


print("Global stiffness matrix:")
A, b = Global_matrix(Elements, n_nodes)

print(A,"\n")



Number of nodes:  6
Number of elements:  4


Elements:
Element 1: [[ 0.   1. ]
 [-0.5  0.5]
 [ 0.6  0.4]]
Element 2: [[ 0.6  0.4]
 [-0.5  0.5]
 [ 0.  -0.2]]
Element 3: [[-0.8  0. ]
 [ 0.  -0.2]
 [-0.5  0.5]]
Element 4: [[ 0.  -0.2]
 [ 1.  -0.1]
 [ 0.6  0.4]]


Global stiffness matrix:
[[ 1.01666667 -0.6         0.         -0.41666667  0.          0.        ]
 [-0.6         1.83913043 -0.58695652 -0.08333333 -0.56884058  0.        ]
 [ 0.         -0.58695652  0.80434783  0.         -0.2173913   0.        ]
 [-0.41666667 -0.08333333  0.          1.86574074 -0.75462963 -0.61111111]
 [ 0.         -0.56884058 -0.2173913  -0.75462963  1.59641707 -0.05555556]
 [ 0.          0.          0.         -0.61111111 -0.05555556  0.66666667]] 



In [50]:
# Dirichlet boundary conditions
a = b = 1  #(-0.5,0.5)
c = d =2  #(-1,1)

#nodes in and out

x, y = nodes[:, 0], nodes[:, 1]
tol = 1e-10                      

on_inner = (np.isclose(np.abs(x), a/2, atol=tol) & (np.abs(y) <= b/2 + tol)) | \
           (np.isclose(np.abs(y), b/2, atol=tol) & (np.abs(x) <= a/2 + tol))

on_outer = (np.isclose(np.abs(x), c/2, atol=tol) & (np.abs(y) <= d/2 + tol)) | \
           (np.isclose(np.abs(y), d/2, atol=tol) & (np.abs(x) <= c/2 + tol))

nodes_in = np.where(on_inner)[0]    
nodes_out = np.where(on_outer)[0]
    

print("Nodes inside the boundary:", nodes_in)
print("Nodes outside the boundary:", nodes_out)
print("\n")

#Essential boundary nodes and Natural boundary nodes

nodes_essential = np.union1d(nodes_in, nodes_out)
nodes_all = np.arange(n_nodes)
nodes_natural = np.setdiff1d(nodes_all, nodes_essential)

print("Essential boundary nodes:", nodes_essential)
print("Natural boundary nodes:", nodes_natural)
print("\n")  




Nodes inside the boundary: [1]
Nodes outside the boundary: [0 5]


Essential boundary nodes: [0 1 5]
Natural boundary nodes: [2 3 4]




In [51]:
Ae = A[np.ix_(nodes_natural,nodes_essential)]
An = A[np.ix_(nodes_natural,nodes_natural)]

z=np.zeros(n_nodes)
z[nodes_in] = U
z[nodes_out] = 0

ze= z[nodes_essential]

print("Partition Vector z :",z)
print("Essential varible Vector z :",ze)

# b = np.zeros(n_nodes)

#Anzn =-Aeze

rhs = -Ae @ ze
zn = np.linalg.solve(An, rhs)

print("Discretized potential values",zn)





Partition Vector z : [0. 1. 0. 0. 0. 0.]
Essential varible Vector z : [0. 1. 0.]
Discretized potential values [0.89665477 0.29447269 0.61762265]


In [52]:
#laplace Equation

e = 8.854e-12

W = 0.5*e*(z.T @ A @ z)

# C= 2W/U^2
capacitance = (2*W)/(U**2)

print("Electrostatic Energy (W):", W)
print("Capacitance (C):", capacitance)


Electrostatic Energy (W): 8.14183043478261e-12
Capacitance (C): 1.628366086956522e-11
